# Part IV: Eigenvalues and matrix calculus: solutions to the code exercises

Worked solutions to the code exercises from [*Matrices in Practice*](https://trgoodale.github.io/matrix-book/appendices/solutions-4.html), by Tom Goodale.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trgoodale/matrix-book/blob/main/code/python/solutions-part-4.ipynb)

Run the cells in order, from the top: later cells use variables set in earlier ones. Try each exercise yourself first. Change the numbers and run them again to see what happens. The code needs NumPy, SciPy and Matplotlib (and pandas in a few chapters), all of which Colab already has.

Code licensed under the [MIT licence](https://trgoodale.github.io/matrix-book/LICENSE.md).

## Chapter 18: Eigenvalues and eigenvectors

### Exercise 7

Three equal masses are joined in a line by four equal springs, with the outer springs fixed to walls. Their vibrations are governed by the matrix $\mathbf{K} = \begin{pmatrix}2 & -1 & 0 \\ -1 & 2 & -1 \\ 0 & -1 & 2\end{pmatrix}$: the natural frequencies are proportional to the square roots of its eigenvalues, and the eigenvectors are the mode shapes, the patterns in which the masses move together. Find the eigenvalues and eigenvectors, and describe each mode shape in words.

In [ ]:
import numpy as np

K = np.array([[2, -1, 0], [-1, 2, -1], [0, -1, 2]])
values, vectors = np.linalg.eigh(K)        # K is symmetric
print(values)
print(np.round(vectors, 4))

The eigenvalues are $2 - \sqrt{2} \approx 0.586$, $2$ and $2 + \sqrt{2} \approx 3.414$, with eigenvectors proportional to $(1, \sqrt{2}, 1)$, $(-1, 0, 1)$ and $(1, -\sqrt{2}, 1)$ (software may flip their signs).

- Lowest frequency: all three masses move the same way together, the middle one furthest.
- Middle frequency: the outer masses move in opposite directions and the middle one stays still.
- Highest frequency: the outer masses move together and the middle one moves the opposite way, by more.

The more the neighbouring masses move against each other, the more the springs between them stretch, and the higher the frequency.

## Chapter 19: Diagonalisation and matrix powers

### Exercise 7

A Leslie matrix models a population divided into age groups. For a species with three age groups (young, adult, old), suppose each adult has 1.5 young per year and each old animal 1, 60% of the young survive to become adults, and 70% of adults survive to become old:
$$
\mathbf{L} = \begin{pmatrix}0 & 1.5 & 1 \\ 0.6 & 0 & 0 \\ 0 & 0.7 & 0\end{pmatrix}.
$$
Find the dominant eigenvalue and its eigenvector. What is the long-run growth rate of the population, and what fraction of the animals is in each age group in the long run? Check by applying $\mathbf{L}$ 50 times to a starting population.

In [ ]:
import numpy as np

L = np.array([[0, 1.5, 1], [0.6, 0, 0], [0, 0.7, 0]])
values, vectors = np.linalg.eig(L)
i = np.argmax(values.real)                 # the dominant eigenvalue is real
lam, v = values[i].real, vectors[:, i].real
print(lam)
print(v / v.sum())                         # scale to fractions that add to 1

x = np.array([100.0, 0, 0])
for k in range(50):
    x = L @ x
print(x / x.sum())

The dominant eigenvalue is about $1.128$, so in the long run the population grows by about 12.8% a year. The other two eigenvalues are complex, with absolute value about $0.61$, so their modes die away. In the long run about 53.7% of the animals are young, 28.6% adult and 17.7% old, and 50 steps from a starting population of young animals only has already reached this distribution.

## Chapter 20: Symmetric and positive definite matrices

### Exercise 7

At a point inside a machine part, the stress matrix in MPa is
$$
\mathbf{\sigma} = \begin{pmatrix}80 & 20 & 0 \\ 20 & 40 & -10 \\ 0 & -10 & -30\end{pmatrix}.
$$
Find the principal stresses and the direction of the largest one. Check that the principal directions are perpendicular.

In [ ]:
import numpy as np

sigma = np.array([[80, 20, 0], [20, 40, -10], [0, -10, -30]])
values, Q = np.linalg.eigh(sigma)
print(values)                              # principal stresses, smallest first
print(Q[:, 2])                             # direction of the largest
print(np.round(Q.T @ Q, 10))               # the identity, so Q is orthogonal

The principal stresses are about $88.4$ MPa (tension), $33.1$ MPa (tension) and $-31.5$ MPa (compression). The largest acts along $\pm(0.921, 0.387, -0.033)$, mostly along the $x$ axis, tilted towards $y$. $\mathbf{Q}^{\mathsf{T}}\mathbf{Q} = \mathbf{I}$, up to rounding, confirms that the principal directions are perpendicular.

## Chapter 21: The singular value decomposition and PCA

### Exercise 7

Generate data for 200 people with two closely related measurements, for example with `rng = np.random.default_rng(1)`, `h = rng.normal(170, 10, 200)` (heights in cm) and `s = h + rng.normal(0, 4, 200)` (arm spans in cm). Find the principal components and the fraction of the variance explained by each, and interpret the first component.

In [ ]:
import numpy as np

rng = np.random.default_rng(1)
h = rng.normal(170, 10, 200)               # heights in cm
s = h + rng.normal(0, 4, 200)              # arm spans in cm
X = np.column_stack([h, s])
Xc = X - X.mean(axis=0)
U, sv, Vt = np.linalg.svd(Xc, full_matrices=False)
print(Vt)                                  # rows are the principal components
print(sv**2 / np.sum(sv**2))               # fraction of variance explained

In MATLAB, the same steps are `X = [h, s]; Xc = X - mean(X); [~, S, V] = svd(Xc, 'econ'); diag(S).^2 / sum(diag(S).^2)`. MATLAB's random numbers differ from NumPy's, so the results differ slightly.

The first principal component is about $\pm(0.67, 0.74)$ and explains about 97% of the variance: it is overall body size, with height and arm span increasing together. The second, perpendicular to it, explains the remaining 3%; it measures how much longer or shorter a person's arm span is than you would expect from their height. One number per person, the score along the first component, captures almost all the information in the two measurements.

## Chapter 22: Derivatives with vectors and matrices

### Exercise 7

Use gradient descent to minimise $f(\mathbf{x}) = \tfrac{1}{2}\mathbf{x}^{\mathsf{T}}\mathbf{A}\mathbf{x}$ with $\mathbf{A} = \begin{pmatrix}10 & 0 \\ 0 & 1\end{pmatrix}$, starting from $(1, 1)$, with step sizes $\alpha = 0.1$, $0.18$ and $0.21$. For each, count the steps until $\lVert\mathbf{x}\rVert < 10^{-6}$, or report that it fails. Explain the results using the eigenvalues of $\mathbf{A}$.

In [ ]:
import numpy as np

A = np.diag([10.0, 1.0])
for alpha in [0.1, 0.18, 0.21]:
    x = np.array([1.0, 1.0])
    k = 0
    while np.linalg.norm(x) >= 1e-6 and k < 1000:
        x = x - alpha * (A @ x)
        k += 1
    print(f"alpha = {alpha:.2f}: {k} steps, |x| = {np.linalg.norm(x):.3g}")

The eigenvalues of $\mathbf{A}$ are 10 and 1, and each step multiplies the two components of $\mathbf{x}$ by $1 - 10\alpha$ and $1 - \alpha$.

- $\alpha = 0.1$: the factors are 0 and $0.9$. The first component vanishes at once, but the second shrinks slowly, and $0.9^k < 10^{-6}$ needs $k = 132$.
- $\alpha = 0.18$: the factors are $-0.8$ and $0.82$, both well inside $(-1, 1)$, so about 70 steps. This is close to the best possible, $\alpha = 2/(10 + 1) \approx 0.182$, which makes both factors $\pm 9/11$.
- $\alpha = 0.21$: the factor $1 - 2.1 = -1.1$ has size greater than 1, so the first component flips sign and grows at every step, and the method diverges.

Convergence needs $\alpha < 2/\lambda_{\max} = 0.2$, and even the best step leaves a factor of $(\kappa - 1)/(\kappa + 1)$ per step, where $\kappa = 10$ is the condition number.

## Chapter 23: Linear ODE systems and the matrix exponential

### Exercise 7

A dose of 100 mg of a drug is swallowed. The amount $x_1$ in the gut and $x_2$ in the blood obey
$$
\dot{x}_1 = -0.5x_1,
\qquad
\dot{x}_2 = 0.5x_1 - 0.2x_2,
$$
with time in hours. Use the matrix exponential to compute $\mathbf{x}(t)$ for $0 \le t \le 20$, plot both amounts, and find when the amount in the blood is greatest and what it is. Check against the exact answer from the eigenvalues.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import expm

A = np.array([[-0.5, 0], [0.5, -0.2]])
t = np.linspace(0, 20, 2001)
X = np.array([expm(A * tk) @ [100, 0] for tk in t])
j = np.argmax(X[:, 1])
print(t[j], X[j, 1])                       # time and amount of the peak

plt.figure(figsize=(6, 3.5))
plt.plot(t, X[:, 0], label="gut")
plt.plot(t, X[:, 1], label="blood")
plt.xlabel("time (hours)")
plt.ylabel("amount (mg)")
plt.legend()
plt.show()

The matrix is triangular, so the eigenvalues are $-0.5$ and $-0.2$. Solving directly, $x_1 = 100e^{-0.5t}$, and
$$
x_2(t) = \frac{100 \times 0.5}{0.5 - 0.2}\left(e^{-0.2t} - e^{-0.5t}\right) = \frac{500}{3}\left(e^{-0.2t} - e^{-0.5t}\right).
$$
This is greatest when $\dot{x}_2 = 0$, that is $0.2e^{-0.2t} = 0.5e^{-0.5t}$, so $e^{0.3t} = 2.5$ and $t = \ln 2.5/0.3 \approx 3.05$ hours, when $x_2 \approx 54.3$ mg. The computed peak agrees, to the resolution of the time grid.

## Part IV: Eigenvalues and matrix calculus review

### Exercise 11

For the system in exercise 5, compute $\mathbf{x}(1)$ in two ways: with the built-in matrix exponential, and with $\mathbf{V}e^{\mathbf{\Lambda}t}\mathbf{V}^{-1}\mathbf{x}(0)$ using the eigenvalues and eigenvectors. Compare both with your formula.

*Uses [the chapter](https://trgoodale.github.io/matrix-book/part-4/23-linear-odes.html) and [the chapter](https://trgoodale.github.io/matrix-book/part-4/19-diagonalisation.html).*

In [ ]:
import numpy as np
from scipy.linalg import expm

A = np.array([[-2.0, 1], [1, -2]])
x0 = np.array([1.0, 0])
t = 1.0

x_expm = expm(A * t) @ x0
lam, V = np.linalg.eigh(A)          # A is symmetric
x_eig = V @ np.diag(np.exp(lam * t)) @ np.linalg.solve(V, x0)
x_formula = 0.5*np.exp(-t)*np.array([1, 1]) + 0.5*np.exp(-3*t)*np.array([1, -1])

for x in (x_expm, x_eig, x_formula):
    print(np.round(x, 6))

All three agree. `expm` doesn't need the eigenvectors, so it also works for matrices that can't be diagonalised.